<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-cnns-in-code.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10 — CNNs in code {.unnumbered}

Three parts, following the book page:

1. **Convolution and images.** Why weight sharing gives translation
   equivariance, then **Flowers102** (102 flower species, only 10
   training images each): a small CNN from scratch, the same CNN with
   data augmentation, and transfer learning from an ImageNet-trained
   ResNet-18. Plus a test of what a CNN is *not* invariant to: rotation.
2. **CNNs on protein sequences.** Days 8-9's subcellular-localization
   dataset and homology-aware split, now with a 1D CNN, and a look inside
   its first layer: the filters behave like learned PWMs (Day 5).
3. **Contact maps.** A real contact map computed from hemoglobin's 3D
   structure (Day 7): the kind of 2D image that deep CNNs learned to
   predict from sequence alignments between 2016 and 2018.

**Runtime.** Part 1 trains two CNNs for `EPOCHS = 100` epochs each. On a
GPU (Colab: *Runtime → Change runtime type → T4 GPU*) that takes about
2-3 minutes per run. On a CPU it takes several seconds per epoch: set
`EPOCHS = 30` for a quicker, lower-accuracy run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
import os, tempfile
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
device = "cuda" if torch.cuda.is_available() else "cpu"
FIGS = "../figs" if os.path.isdir("../figs") else "."
print("device:", device)

# Part 1 — Convolution and images

## Part 1: why convolution gives translation equivariance (and, with pooling, invariance)

A 1D convolution slides the *same* small kernel across every position of its
input. If the input signal is shifted, the kernel sees the same local pattern,
just later (or earlier) -- so the output feature map should shift by exactly
the same amount. That is **equivariance**: `conv(shift(x)) == shift(conv(x))`
(up to the boundary, where the kernel runs off the edge). This is a property
of the architecture itself -- weight sharing -- and holds even for a
completely untrained, randomly-initialized kernel, which is what we use below
so there is no training involved in this demonstration at all.

In [ ]:
conv = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=5, padding=2, bias=False)

# A toy signal: a single sharp "bump" at position 20 in a length-60 sequence.
length = 60
signal = torch.zeros(1, 1, length)
signal[0, 0, 20] = 1.0

shift = 10
shifted_signal = torch.zeros(1, 1, length)
shifted_signal[0, 0, 20 + shift] = 1.0

with torch.no_grad():
    out = conv(signal)[0, 0].numpy()
    out_shifted = conv(shifted_signal)[0, 0].numpy()

# Compare: shifting the OUTPUT of the original by `shift` positions should
# match the output computed from the ALREADY-shifted input (away from the
# boundary effects near the edges where the kernel runs off the sequence).
out_rolled = np.roll(out, shift)
interior = slice(10, 50)  # stay away from both edges
max_abs_diff = np.max(np.abs(out_rolled[interior] - out_shifted[interior]))
print(f"max |shift(conv(x)) - conv(shift(x))| over the interior region: {max_abs_diff:.2e}")
print("equivariant (matches to floating-point precision away from the edges):", max_abs_diff < 1e-6)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(signal[0, 0].numpy(), label="input (bump at 20)")
axes[0].plot(shifted_signal[0, 0].numpy(), label=f"input shifted by {shift}")
axes[0].set_title("Two inputs: the same bump, shifted")
axes[0].legend()
axes[1].plot(out, label="conv(original input)")
axes[1].plot(out_shifted, label="conv(shifted input)")
axes[1].plot(out_rolled, "--", label="shift(conv(original input))", alpha=0.7)
axes[1].set_title("The shifted input's output matches the original output, shifted")
axes[1].legend()
axes[1].set_xlabel("position")
plt.tight_layout()
plt.savefig("../figs/day10-conv-equivariance.png", dpi=150)
plt.show()

**From equivariance to invariance.** Equivariance alone still leaves the bump
at a different position in the output feature map -- useful, but not yet
"the same answer regardless of position." **Global pooling** (taking the
max, or the mean, over the whole feature map) removes the position
information entirely: the *value* of the peak is preserved by shifting, so
its max is too, regardless of where it sits. That combination --
weight-shared convolution (equivariant) followed by global pooling
(position-independent) -- is exactly what gives a convolutional network
translation invariance, and it requires no training to exist as a structural
property of the architecture.

In [ ]:
pooled_original = out.max()
pooled_shifted = out_shifted.max()
print(f"max-pooled output, original input:  {pooled_original:.6f}")
print(f"max-pooled output, shifted input:   {pooled_shifted:.6f}")
print("identical after pooling:", np.isclose(pooled_original, pooled_shifted))

## Flowers102: very little data per class

The Oxford Flowers102 dataset (Nilsback & Zisserman, 2008) has 102 flower
species. The official split gives only **10 training images per class**
(1,020 in total), 1,020 validation and 6,149 test images. That is a
realistic situation in biology: many classes, few labelled examples. We
evaluate on a fixed random subset of 2,000 test images to keep the runtime
down.

In [ ]:
EPOCHS = 100          # 30 is fine for a quick CPU run (lower accuracy)
DATA_DIR = os.path.join(tempfile.gettempdir(), "kb8029_flowers")   # not next to this notebook
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])     # ImageNet channel statistics

plain_tf = T.Compose([T.Resize(72), T.CenterCrop(64), T.ToTensor(), norm])
aug_tf = T.Compose([T.RandomResizedCrop(64, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), norm])
flowers = lambda split, tf: torchvision.datasets.Flowers102(DATA_DIR, split=split, transform=tf, download=True)

train_plain, train_aug = flowers("train", plain_tf), flowers("train", aug_tf)
val_set, test_full = flowers("val", plain_tf), flowers("test", plain_tf)
test_idx = np.random.RandomState(0).choice(len(test_full), 2000, replace=False)
test_set = Subset(test_full, test_idx)
print(f"train {len(train_plain)}  val {len(val_set)}  test {len(test_full)} (we use {len(test_set)})")
labels = np.array(train_plain._labels)
print("training images per class:", np.bincount(labels).min(), "to", np.bincount(labels).max())

**Data augmentation** creates a new, slightly changed version of every
training image in every epoch: a random crop and zoom, a mirror image, a
small rotation, some change of brightness and colour. The label stays the
same. It is a cheap way of telling the network which changes should *not*
matter.

In [ ]:
raw_train = torchvision.datasets.Flowers102(DATA_DIR, split="train", download=True)
img0, _ = raw_train[0]
show_tf = T.Compose([T.RandomResizedCrop(160, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                     T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2)])
torch.manual_seed(1)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
axes[0].imshow(T.Compose([T.Resize(180), T.CenterCrop(160)])(img0)); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(show_tf(img0)); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIGS}/day10-augmentation.png", dpi=150, bbox_inches="tight"); plt.show()

### A small CNN, trained from scratch

Four blocks of convolution (3×3) → batch normalization → ReLU → 2×2 max
pooling, then global average pooling and a linear layer to the 102
classes. It has about 255,000 weights. We train it twice, on the same
images, without and with augmentation.

In [ ]:
def small_cnn(n_classes=102):
    block = lambda i, o: [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2)]
    return nn.Sequential(*block(3, 32), *block(32, 64), *block(64, 128), *block(128, 128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))

def evaluate(model, dataset):
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=256, num_workers=2):
            correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return correct / len(dataset)

def train_cnn(train_set, epochs=EPOCHS, seed=0):
    torch.manual_seed(seed)
    model = small_cnn().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2,
                        generator=torch.Generator().manual_seed(seed))
    hist, best = [], (0.0, None)
    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            opt.zero_grad()
            nn.functional.cross_entropy(model(x.to(device)), y.to(device)).backward()
            opt.step()
        tr, va = evaluate(model, train_plain), evaluate(model, val_set)
        hist.append((tr, va))
        if va > best[0]:
            best = (va, {k: v.clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best[1])                      # early stopping on validation accuracy
    return model, hist

results = {}
for name, ds in [("from scratch", train_plain), ("from scratch + augmentation", train_aug)]:
    model, hist = train_cnn(ds)
    results[name] = dict(hist=hist, test=evaluate(model, test_set),
                         n=sum(p.numel() for p in model.parameters()))
    best_ep = int(np.argmax([v for _, v in hist]))
    print(f"{name:30s} weights {results[name]['n']:,}  best val {hist[best_ep][1]:.3f} (epoch {best_ep + 1}), "
          f"train acc then {hist[best_ep][0]:.3f}  ->  TEST {results[name]['test']:.3f}")

### Transfer learning from ImageNet

The alternative that took over the field: start from a network that has
already learned general visual features (edges, textures, parts of
objects) from 1.2 million ImageNet photos, keep those layers **frozen**,
and train only a new last layer on our flowers. Here: ResNet-18 (He et
al., 2016) with its ImageNet weights, used as a fixed feature extractor
(512 numbers per image), and a logistic-regression head on top.

In [ ]:
from sklearn.linear_model import LogisticRegression
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
resnet = torchvision.models.resnet18(weights=weights).to(device).eval()
backbone = nn.Sequential(*list(resnet.children())[:-1], nn.Flatten())   # everything except the last layer
tf224 = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

def features(dataset):
    F, Y = [], []
    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=64, num_workers=2):
            F.append(backbone(x.to(device)).cpu()); Y.append(y)
    return torch.cat(F).numpy(), torch.cat(Y).numpy()

F_tr, Y_tr = features(flowers("train", tf224))
F_va, Y_va = features(flowers("val", tf224))
F_te, Y_te = features(Subset(flowers("test", tf224), test_idx))
head = LogisticRegression(max_iter=3000).fit(F_tr, Y_tr)
results["transfer (frozen ResNet-18)"] = dict(val=head.score(F_va, Y_va), test=head.score(F_te, Y_te))
print(f"feature vector per image: {F_tr.shape[1]} numbers")
print(f"transfer learning: val {results['transfer (frozen ResNet-18)']['val']:.3f}  "
      f"TEST {results['transfer (frozen ResNet-18)']['test']:.3f}")

In [ ]:
plt.figure(figsize=(6.5, 4))
for name, color in [("from scratch", "#8a8a8a"), ("from scratch + augmentation", "#d9822b")]:
    tr, va = zip(*results[name]["hist"])
    plt.plot(np.arange(1, len(va) + 1), va, color=color, label=f"{name}: validation")
    plt.plot(np.arange(1, len(tr) + 1), tr, color=color, ls="--", lw=1, label=f"{name}: training")
plt.axhline(results["transfer (frozen ResNet-18)"]["val"], color="#2f6db5", label="transfer learning: validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1.02)
plt.title("Flowers102, 10 training images per class")
plt.legend(fontsize=8, loc="center right")
plt.savefig(f"{FIGS}/day10-flowers-curves.png", dpi=150, bbox_inches="tight"); plt.show()
for name, r in results.items():
    print(f"{name:30s} TEST accuracy {r['test']:.3f}")

## What a CNN is (and isn't) invariant to

Convolution plus pooling makes a CNN largely insensitive to *where* an
object is. Nothing in the architecture makes it insensitive to
*rotation*. A test with the ImageNet-trained ResNet-18 itself (ImageNet
has a "daisy" class): find a test image it confidently calls a daisy,
then shift it and rotate it. Every version is cropped the same way after
rotating, so no black corners enter the picture.

In [ ]:
categories = weights.meta["categories"]
raw_test = torchvision.datasets.Flowers102(DATA_DIR, split="test", download=True)
to_input = T.Compose([T.ToTensor(), norm])
def prepare(img, angle=0, shift=0):
    img = T.Resize(256)(img)
    img = T.functional.rotate(img, angle)
    img = T.CenterCrop(176)(img)                  # small enough that rotated corners are cut away
    img = T.functional.affine(img, angle=0, translate=(shift, shift), scale=1.0, shear=0)
    return T.Resize(224)(img)

daisy = categories.index("daisy")
for k in range(len(raw_test)):
    img, _ = raw_test[k]
    with torch.no_grad():
        p = resnet(to_input(prepare(img))[None].to(device)).softmax(1)[0]
    if p.argmax().item() == daisy and p[daisy] > 0.8:
        break
print(f"test image {k}: P(daisy) = {p[daisy]:.3f}")

rows = []
for kind, values in [("rotate (degrees)", [0, 15, 30, 45, 90, 180]), ("shift (pixels)", [0, 10, 20, 30])]:
    for v in values:
        im = prepare(img, angle=v) if kind.startswith("rotate") else prepare(img, shift=v)
        with torch.no_grad():
            q = resnet(to_input(im)[None].to(device)).softmax(1)[0]
        rows.append((kind, v, im, q[daisy].item(), categories[q.argmax().item()]))
        print(f"{kind:17s} {v:4d}: P(daisy) = {q[daisy]:.3f}   top class: {categories[q.argmax().item()]}")

fig, axes = plt.subplots(2, 6, figsize=(12, 4.6))
for ax in axes.flat: ax.axis("off")
for ax, (kind, v, im, pd, top) in zip(list(axes[0]) + list(axes[1][:4]), rows):
    ax.imshow(im)
    ax.set_title(f"{'rot' if kind.startswith('rotate') else 'shift'} {v}\nP(daisy)={pd:.2f}\n{top}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIGS}/day10-rotation-shift.png", dpi=150, bbox_inches="tight"); plt.show()

Shifts barely change the answer, while rotations can change it
completely. A daisy is roughly radially symmetric, so 90° and 180°
rotations may still look like daisies; intermediate angles need not.
This is why rotation is a standard *augmentation*. The network has to be
shown rotated examples, because the architecture does not provide
rotation invariance for free.

# Part 2 — CNNs on protein sequences

## Part 2: the subcellular-localization case study, this time with a 1D CNN

Same dataset, same encoding, same homology-aware train/val/test split as Days 8-9 (fetched
live from UniProt again here, with the same fixed random seeds, so it should
reproduce the same 700-protein, 5-class, 486/105/109 homology-aware split) -- but this time
the classifier is a 1D convolutional network over the sequence, instead of a
flat fully-connected one, directly demonstrating the "1D convolution over a
sequence" idea from the book page.

In [ ]:
import re
import requests

UNIPROT_CLASSES = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
}
CLASS_NAMES = list(UNIPROT_CLASSES.keys())

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                "format": "tsv", "size": size},
        timeout=60,
    )
    r.raise_for_status()
    rows = []
    for line in r.text.strip().split("\n")[1:]:
        parts = line.split("\t")
        if len(parts) == 3:
            rows.append(tuple(parts))
    return rows

def is_unambiguous_single_location(location_text, target):
    location_text = re.sub(r"Note=.*", "", location_text)
    location_text = re.sub(r"\{[^}]*\}", "", location_text)
    if "Isoform" in location_text:
        return False
    terms = set()
    for statement in [s.strip() for s in location_text.split(".") if s.strip()]:
        body = statement.split(":", 1)[-1] if ":" in statement else statement
        top_term = re.split(r"[,;]", body)[0].strip().rstrip(".")
        if top_term:
            terms.add(top_term)
    return terms == {target}

entries_by_class = {name: [] for name in CLASS_NAMES}
for name, code in UNIPROT_CLASSES.items():
    rows = fetch_uniprot(code, size=500)
    kept = [(acc, seq) for (acc, seq, loc) in rows if is_unambiguous_single_location(loc, name)]
    entries_by_class[name] = kept
    print(f"{name}: fetched {len(rows)}, unambiguous single-location {len(kept)}")

In [ ]:
N_PER_CLASS = min(len(v) for v in entries_by_class.values())
print("balancing every class to", N_PER_CLASS, "sequences")

rng = np.random.RandomState(0)
balanced_accs, balanced_seqs, balanced_labels = [], [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    pool = entries_by_class[class_name]
    for i in rng.choice(len(pool), N_PER_CLASS, replace=False):
        balanced_accs.append(pool[i][0])
        balanced_seqs.append(pool[i][1])
        balanced_labels.append(class_idx)

print("total balanced dataset:", len(balanced_seqs), "sequences,", len(CLASS_NAMES), "classes")
print("(Day 8/9 reported 700 sequences, 140/class -- checking this matches)")

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
SEQ_LEN = 150

def one_hot_encode(sequence, length=SEQ_LEN):
    encoded = np.zeros((length, len(AMINO_ACIDS)), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        if residue in AA_TO_INDEX:
            encoded[position, AA_TO_INDEX[residue]] = 1.0
    return encoded  # (SEQ_LEN, 20) -- NOT flattened this time, a CNN wants the 2D shape

X = np.stack([one_hot_encode(s) for s in balanced_seqs])       # (N, 150, 20)
X = X.transpose(0, 2, 1)                                        # (N, 20, 150): channels=amino acids, length=position
y = np.array(balanced_labels)
print("X shape:", X.shape, " (N, channels=20 amino acids, length=150 positions)")
print("y shape:", y.shape)

In [ ]:
import os, shutil, subprocess, tempfile

# Day 8's homology-aware split: MMseqs2 clusters at 30% identity, whole clusters per split
CLUSTER_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/day08-subcell-mmseqs-30.tsv"
LOCAL_TSV = "data/day08-subcell-mmseqs-30.tsv"

def mmseqs_clusters(accs, seqs, min_seq_id=0.3):
    tmp = tempfile.mkdtemp()
    with open(f"{tmp}/in.fasta", "w") as f:
        for a, s in zip(accs, seqs):
            f.write(f">{a}\n{s}\n")
    subprocess.run(["mmseqs", "easy-cluster", f"{tmp}/in.fasta", f"{tmp}/clu", f"{tmp}/work",
                    "--min-seq-id", str(min_seq_id), "-c", "0.5", "-v", "1"],
                   check=True, stdout=subprocess.DEVNULL)
    pairs = [line.split("\t") for line in open(f"{tmp}/clu_cluster.tsv").read().split("\n") if line]
    shutil.rmtree(tmp)
    return {member: rep for rep, member in pairs}

def load_precomputed_clusters():
    text = open(LOCAL_TSV).read() if os.path.exists(LOCAL_TSV) else requests.get(CLUSTER_URL, timeout=30).text
    pairs = [line.split("\t") for line in text.strip().split("\n")[1:]]
    return {member: rep for rep, member in pairs}

if shutil.which("mmseqs"):
    member_to_rep = mmseqs_clusters(balanced_accs, balanced_seqs)
    print("clustered with local MMseqs2 (30% identity, 50% coverage)")
else:
    member_to_rep = load_precomputed_clusters()
    print("mmseqs not found -- loaded the precomputed MMseqs2 clustering")

# proteins missing from the clustering (e.g. if UniProt changed) become their own cluster
reps = [member_to_rep.get(a, a) for a in balanced_accs]
rep_ids = {r: i for i, r in enumerate(sorted(set(reps)))}
groups = np.array([rep_ids[r] for r in reps])
sizes = np.bincount(groups)
print(f"{len(y)} proteins -> {len(sizes)} clusters; "
      f"{(sizes > 1).sum()} clusters have more than one member ({sizes[sizes > 1].sum()} proteins), largest has {sizes.max()}")

def homology_split(y, groups, fractions=(0.70, 0.15, 0.15), seed=0):
    '''Assign whole clusters, in random order, to train/val/test: each cluster
    goes to the split furthest below its target share of that cluster's
    classes (ties broken at random). Returns three index arrays.'''
    rng = np.random.RandomState(seed)
    n_classes = y.max() + 1
    target = np.outer(fractions, np.bincount(y, minlength=n_classes))   # (3 splits, n_classes)
    have = np.zeros_like(target)
    assignment = {}
    for g in rng.permutation(np.unique(groups)):
        cls_counts = np.bincount(y[groups == g], minlength=n_classes)
        deficit = ((target - have) * (cls_counts > 0)).sum(axis=1) / target.sum(axis=1)
        best = np.flatnonzero(deficit == deficit.max())
        split = int(rng.choice(best))
        assignment[g] = split
        have[split] += cls_counts
    split_of = np.array([assignment[g] for g in groups])
    return [np.where(split_of == k)[0] for k in range(3)]

tr_h, va_h, te_h = homology_split(y, groups)
X_train, y_train = X[tr_h], y[tr_h]
X_val, y_val = X[va_h], y[va_h]
X_test, y_test = X[te_h], y[te_h]
print(f"train: {len(X_train)}   val: {len(X_val)}   test: {len(X_test)}")
print("(Day 8/9 reported 486 / 105 / 109 -- checking this matches)")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t   = torch.tensor(y_val,   dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

### A 1D-CNN classifier

Two convolutional layers (kernel size 7, i.e. each filter looks at a 7-residue
window) with ReLU and max-pooling, then global average pooling over the
remaining sequence length (making the classifier's decision insensitive to
*where* along the sequence a signal was detected -- exactly Part 1's
equivariance-then-pooling argument, now inside a real classifier), then a
linear layer to the 5 classes.

In [ ]:
class LocalizationCNN(nn.Module):
    def __init__(self, n_classes=5, n_channels=20):
        super().__init__()
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=7, padding=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=7, padding=3)
        self.pool = nn.MaxPool1d(2)
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):                      # x: (batch, 20, 150)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = torch.adaptive_avg_pool1d(x, 1).squeeze(-1)   # global average pool -> (batch, 64)
        return self.classifier(x)               # raw logits, (batch, n_classes)


cnn_model = LocalizationCNN()
n_params = sum(p.numel() for p in cnn_model.parameters())
print(cnn_model)
print("total trainable parameters:", n_params)
print("(Day 8/9's flat MLP had ~192,389 parameters -- for comparison)")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_acc = 0.0
best_state = None
history = {"train_acc": [], "val_acc": []}

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(dim=1)
    return (preds == y).float().mean().item()

for epoch in range(100):
    cnn_model.train()
    optimizer.zero_grad()
    logits = cnn_model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    train_acc = accuracy(cnn_model, X_train_t, y_train_t)
    val_acc = accuracy(cnn_model, X_val_t, y_val_t)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        best_state = {k: v.clone() for k, v in cnn_model.state_dict().items()}

cnn_model.load_state_dict(best_state)
print(f"best validation accuracy {best_val_acc:.3f} at epoch {best_epoch} of 100 (early-stopped, same practice as Day 9)")
print(f"final train accuracy at that epoch: {history['train_acc'][best_epoch]:.3f}")

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.plot(history["train_acc"], label="train accuracy")
plt.plot(history["val_acc"], label="validation accuracy")
plt.axvline(best_epoch, linestyle="--", color="gray", label=f"best epoch ({best_epoch})")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("1D-CNN training on subcellular localization")
plt.legend()
plt.tight_layout()
plt.savefig("../figs/day10-cnn-training-curve.png", dpi=150)
plt.show()

### Evaluating on the held-out test set, with the same metrics as Day 8/9

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, matthews_corrcoef
import torch.nn.functional as F

cnn_model.eval()
with torch.no_grad():
    test_logits = cnn_model(X_test_t)
    test_probs = F.softmax(test_logits, dim=1).numpy()
    test_preds = test_logits.argmax(dim=1).numpy()

cnn_test_acc = (test_preds == y_test).mean()
precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_preds, average="macro", zero_division=0)
cnn_mcc = matthews_corrcoef(y_test, test_preds)
cnn_auroc = roc_auc_score(y_test, test_probs, multi_class="ovr", average="macro")

print(f"1D-CNN  test accuracy:  {cnn_test_acc:.3f}")
print(f"1D-CNN  macro F1:       {f1:.3f}")
print(f"1D-CNN  MCC:            {cnn_mcc:.3f}")
print(f"1D-CNN  macro AUROC:    {cnn_auroc:.3f}")
print()
print("For comparison, on the SAME homology-aware test set (Days 8-9):")
print("  flat MLP, untrained (Day 8, validation set): accuracy 0.181, macro F1 0.124, MCC -0.027, macro AUROC 0.474")
print("  flat MLP, trained, early-stopped (Day 9):    accuracy 0.514, macro F1 0.493, MCC 0.394, macro AUROC 0.831")
print("  logistic regression, no hidden layer (Day 9): accuracy 0.569,                 MCC 0.461, macro AUROC 0.849")

## Inside the first layer: filters are learned PWMs

Each first-layer filter is a 20 × 7 matrix of weights: one weight per
amino acid per position in a 7-residue window. That is exactly the shape
of a position weight matrix (Day 5), except that the network learned it
from the class labels instead of from an alignment. Which filters matter
for **secreted** proteins? For every filter, compare its strongest
response along the sequence in secreted vs. all other proteins, then look
at the most selective ones.

In [ ]:
from scipy.stats import pearsonr
KD = {'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
      'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2}
kd = np.array([KD[a] for a in AMINO_ACIDS])

cnn_model.eval()
with torch.no_grad():
    act = torch.relu(cnn_model.conv1(torch.tensor(X, dtype=torch.float32)))   # (proteins, filters, positions)
peak_value = act.max(dim=2).values.numpy()
peak_pos = act.argmax(dim=2).numpy()
is_sec = (y == CLASS_NAMES.index("Secreted"))
selectivity = peak_value[is_sec].mean(0) - peak_value[~is_sec].mean(0)
W1 = cnn_model.conv1.weight.detach().numpy()                                  # (filters, 20 amino acids, 7 positions)

print("the 5 filters most selective for secreted proteins:")
for f in np.argsort(-selectivity)[:5]:
    r, p = pearsonr(W1[f].mean(axis=1), kd)
    print(f"  filter {f:2d}: correlation of its amino-acid weights with Kyte-Doolittle hydrophobicity r = {r:.2f} "
          f"(p = {p:.0e}); peak position median {np.median(peak_pos[is_sec, f]):.0f} in secreted, "
          f"{np.median(peak_pos[~is_sec, f]):.0f} in other proteins")
all_r = [pearsonr(W1[f].mean(axis=1), kd)[0] for f in range(W1.shape[0])]
print(f"for comparison, all {W1.shape[0]} filters: mean r = {np.mean(all_r):.2f}")

In [ ]:
best = int(np.argmax(selectivity))
order = np.argsort(-kd)                                   # most hydrophobic amino acids at the top
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.6]})
lim = np.abs(W1[best]).max()
im = a1.imshow(W1[best][order], cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
a1.set_yticks(range(20)); a1.set_yticklabels([AMINO_ACIDS[i] for i in order], fontsize=8)
a1.set_xticks(range(7)); a1.set_xticklabels(range(1, 8)); a1.set_xlabel("position in the 7-residue window")
a1.set_title(f"filter {best}: weights (hydrophobic residues at top)", fontsize=10)
plt.colorbar(im, ax=a1, fraction=0.05)
bins = np.arange(0, SEQ_LEN + 1, 5)
a2.hist(peak_pos[is_sec, best], bins=bins, alpha=0.7, color="#c0392b", label="secreted", density=True)
a2.hist(peak_pos[~is_sec, best], bins=bins, alpha=0.5, color="#8a8a8a", label="other classes", density=True)
a2.set_xlabel("residue position of the filter's strongest response"); a2.set_ylabel("density")
a2.set_title("where along the sequence it fires", fontsize=10); a2.legend()
plt.tight_layout(); plt.savefig(f"{FIGS}/day10-signal-filter.png", dpi=150, bbox_inches="tight"); plt.show()

Nobody told the network about signal peptides. It was given one-hot
sequences and five labels. Yet its most secreted-specific filters weight
hydrophobic residues (L, I, V, F...) positively and charged ones
negatively, and they fire around residues 10-15: the hydrophobic core of
a signal peptide (Day 11 returns to signal peptides in detail). DeepBind
(Alipanahi et al., 2015) made the same observation for DNA: the
first-layer filters of CNNs trained on binding data match known
transcription-factor motifs.

# Part 3 — Contact maps: images made from protein structures

A **contact map** marks which residue pairs are close in 3D: here, pairs
whose Cβ atoms (Cα for glycine) are within 8 Å, the standard CASP
definition. Computed from the real crystal structure of hemoglobin
(PDB 4HHB, chain B = β-globin, Day 7), with the helices from the PDB
file's own annotation drawn along the axes:

In [ ]:
import io, requests
from Bio.PDB import PDBParser
pdb_text = requests.get("https://files.rcsb.org/download/4HHB.pdb", timeout=60).text
chain = PDBParser(QUIET=True).get_structure("4HHB", io.StringIO(pdb_text))[0]["B"]
residues = [r for r in chain if r.id[0] == " "]
coords = np.array([(r["CB"] if "CB" in r else r["CA"]).coord for r in residues])
dist = np.linalg.norm(coords[:, None] - coords[None], axis=2)
contact = dist < 8.0
L = len(residues)
i, j = np.triu_indices(L, 6)
sep = j - i
print(f"{L} residues; contacts with |i-j| >= 6: {contact[i, j].sum()}, of which long-range (|i-j| >= 24): "
      f"{contact[i, j][sep >= 24].sum()}, medium-range (12-23): {contact[i, j][(sep >= 12) & (sep < 24)].sum()}")
helices = [(int(l[21:25]), int(l[33:37])) for l in pdb_text.splitlines() if l.startswith("HELIX") and l[19] == "B"]
print("helices (residue ranges):", helices)

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.8))
ax.imshow(contact, cmap="Greys", origin="upper", extent=(0.5, L + 0.5, L + 0.5, 0.5))
for k, (a, b) in enumerate(helices):
    for rect in [plt.Rectangle((a, -3), b - a, 2.5), plt.Rectangle((-3, a), 2.5, b - a)]:
        rect.set_color("#c0392b"); rect.set_clip_on(False); ax.add_patch(rect)
ax.set_xlim(-3.5, L + 0.5); ax.set_ylim(L + 0.5, -3.5)
ax.set_xlabel("residue"); ax.set_ylabel("residue")
ax.set_title("β-globin (4HHB chain B): Cβ-Cβ < 8 Å\nred bars: α-helices", fontsize=10)
plt.savefig(f"{FIGS}/day10-hbb-contact-map.png", dpi=150, bbox_inches="tight"); plt.show()

The thick band along the diagonal is each helix contacting itself. The
off-diagonal blocks are helix packing against helix. Those long-range
contacts carry the information about the fold, and they are what
contact-prediction methods try to predict from sequence alone. To a CNN,
a contact map is just an L × L image. Predicting it from pairwise features
of a multiple sequence alignment is the image-to-image problem that deep
residual CNNs cracked around 2016-2018 (book page, Part 3), leading
directly to AlphaFold (Day 12).

## Try it yourself

- In Part 1, change `EPOCHS` or the augmentation (e.g. remove `RandomRotation`)
  and re-run. How much of the augmentation gain survives?
- In Part 2, print the weights of the filter most selective for
  **mitochondrion** instead. Mitochondrial targeting peptides are
  amphipathic and rich in arginine: do you see that?
- In Part 3, use an 8 Å cutoff on Cα atoms instead of Cβ. How many
  long-range contacts do you gain or lose?